In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import pickle
import copy

from imports import *
from config import dir_config, ephys_config
from src.utils import dpca_utils, dpca_plot_utils

In [ ]:
compiled_dir  = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

## Load data

Data loading stays in the notebook. The utils start after you have
`session_metadata`, `neuron_metadata`, `ephys`, and the trial-info dicts.

In [ ]:
session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, "sessions_metadata.csv"))
session_metadata = session_metadata[~session_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"))
neuron_metadata = neuron_metadata[~neuron_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

with open(Path(processed_dir, "glm_hmm_models", "glm_hmm_masked_final.pkl"), "rb") as f:
    glm_hmm = pickle.load(f)
glm_hmm_original = copy.deepcopy(glm_hmm)

with open(Path(processed_dir, "ephys_neuron_wise.pkl"), "rb") as f:
    ephys = pickle.load(f)

## Extract trial info (blocks or glm-hmm states)

In [ ]:
data = glm_hmm["data"]

# HMM states — also flips sign for awayRF sessions in-place on glm_hmm["data"]
# compiled_dir loads reaction_time from trial CSVs (required by get_trial_num)
biased_state_trial_info, unbiased_state_trial_info, state_occupancy = \
    dpca_utils.extract_hmm_state_trial_info(session_metadata, glm_hmm_original, data,
                                            compiled_dir=compiled_dir)

# Blocks from prob_toRF (run after extract_hmm_state_trial_info so awayRF sign flip is applied)
equal_block_trial_info, unequal_block_trial_info = \
    dpca_utils.extract_block_trial_info(data, session_metadata["session_id"])

## Shared setup

In [ ]:
toRF_sessions  = session_metadata.session_id[session_metadata.prior_direction == "toRF"]
awayRF_sessions = session_metadata.session_id[session_metadata.prior_direction == "awayRF"]

alignments = list(ephys_config["alignment_settings_GP"].keys())  # ['baseline','visual','cue','response']
marginalization_keys = ['b', 's', 'c', 't']

condition_dict_states = {
    "state_values": ["biased", "unbiased"],
    "coherences":   [0, 0.06, 0.2, 0.5],
    "choices":      ["awayRF", "toRF"],
}

state_trial_info = {
    "biased":   biased_state_trial_info,
    "unbiased": unbiased_state_trial_info,
}

COH_LABELS = ["0%", "6%", "20%", "50%"]


---
## Example 1 — All neurons, HMM states (baseline)

In [ ]:
_out = Path("../results/dpca_toRF_session_all_neuron_verify")
_out.mkdir(parents=True, exist_ok=True)


In [ ]:
neuron_ids = dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_states, neuron_ids,
    state_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results  = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments, marginalization_keys=marginalization_keys)
projections   = dpca_utils.cross_period_projection(dpca_results, fit_avg, alignments)
time_axes     = dpca_utils.build_time_axes(fit_avg, ephys_config)


Plot variance explained, self-projection and cross-projection

In [ ]:
margs_to_plot = ['b', 's', 'c', 't']
n_components  = 3
fig = dpca_plot_utils.plot_variance_explained(dpca_results, alignments, margs_to_plot, n_components=n_components)
plt.show()


In [ ]:
# Significance masks — binary low (0%,6%) vs high (20%,50%) coherence classifier
# smooth_sigma=10 applies Gaussian smoothing to true scores before thresholding against null.

significance_masks = {}
for alignment in alignments:
    dpca_model = dpca_results[alignment]["model"]
    print(f"Computing significance masks for {alignment} alignment...")
    significance_masks[alignment], _, _ = \
        dpca_utils.dpca_significance_analysis(
            copy.deepcopy(dpca_model), fit_avg[alignment], fit_tw[alignment],
            n_shuffles=100, n_splits=50, n_consecutive=1,
            keys=['b', 's', 'c'],
            key_groups={'s': [[0, 1], [2, 3]]},
            smooth_sigma=10,
        )

# significance_masks[alignment][marg_key] → bool array (n_components, n_time)


In [ ]:
# Significant bin counts per alignment / marginalization (PC1)
for alignment in alignments:
    for key in ['b', 's', 'c']:
        n_sig = significance_masks[alignment][key][0].sum()
        print(f"  {alignment}/{key}: {n_sig} significant bins (PC1)")


In [ ]:
PC = 0
margs_to_plot = ['b', 's', 'c']

fig = dpca_plot_utils.plot_self_projection(
    projections, time_axes, alignments, margs_to_plot,
    significance_masks=significance_masks, PC=PC,
    coh_labels=COH_LABELS,
)
plt.show()


In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
with open(_out / "significance_masks.pkl", "wb") as _f:
    pickle.dump(significance_masks, _f)

fig = dpca_plot_utils.plot_variance_explained(dpca_results, alignments, ['b', 's', 'c', 't'], n_components=3)
fig.savefig(_out / "variance_explained.png", dpi=150, bbox_inches="tight")
plt.close(fig)

PC = 0
fig = dpca_plot_utils.plot_self_projection(
    projections, time_axes, alignments, ['b', 's', 'c'],
    significance_masks=significance_masks, PC=PC, coh_labels=COH_LABELS,
)
fig.savefig(_out / "self_projection.png", dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"Saved → {_out}")


### Error trials — project outcome=0 through the fitted axes

In [ ]:
# Build error-trial matrix (outcome=0) and project through axes fitted on correct trials above.
avg_error, _ = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_states, neuron_ids,
    state_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states", outcome=0,
)

# Align to the same cleaned timepoints as fit_avg
error_proj = {a: {} for a in alignments}
for alignment in alignments:
    n_time_fit = fit_avg[alignment].shape[-1]
    X_err = avg_error[alignment]
    X_err_trimmed = X_err[..., -n_time_fit:] if alignment == "response" else X_err[..., :n_time_fit]
    _, Z_err = dpca_utils.dpca_transform(dpca_results[alignment]["model"], X_err_trimmed)
    error_proj[alignment][alignment] = Z_err

fig = dpca_plot_utils.plot_self_projection(
    error_proj, time_axes, alignments, ['b', 's', 'c'],
    significance_masks=None, PC=0, coh_labels=COH_LABELS,
    title="Self-projection: PC1 — error trials (outcome=0)",
)
plt.show()

fig.savefig(_out / "error_self_projection.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {_out / 'error_self_projection.png'}")


In [ ]:
fit_align = "baseline"
margs_to_plot = ['b', 's', 'c']
fig = dpca_plot_utils.plot_cross_projection(projections, time_axes, alignments, margs_to_plot,
                                            fit_align, coh_labels=COH_LABELS)
plt.show()


---
## Example 2 — Exclude trash and undefined (trash excluded by default; add undefined)

In [ ]:
neuron_ids_no_trash = dpca_utils.get_neuron_ids(
    neuron_metadata, toRF_sessions,
    exclude_cell_types=["trash", "undefined"],
)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_states, neuron_ids_no_trash,
    state_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results_no_undefined = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
projections_no_undefined  = dpca_utils.cross_period_projection(dpca_results_no_undefined, fit_avg, alignments)

#### plot results

In [ ]:
margs_to_plot = ['b', 's', 'c', 't']
fig = dpca_plot_utils.plot_variance_explained(dpca_results_no_undefined, alignments, margs_to_plot, n_components=n_components)
plt.show()

PC = 0
margs_to_plot = ['b', 's', 'c']
fig = dpca_plot_utils.plot_self_projection(projections_no_undefined, time_axes, alignments, margs_to_plot,
                                           PC=PC, coh_labels=COH_LABELS)
plt.show()

fit_align = "baseline"
fig = dpca_plot_utils.plot_cross_projection(projections_no_undefined, time_axes, alignments, margs_to_plot,
                                            fit_align, coh_labels=COH_LABELS)
plt.show()


---
## Example 3 — 10% neuron leave-out (repeated for stability)

Each repeat randomly drops 10% of neurons. Pass the same `rng` seed for
reproducibility, or a different seed per repeat for a stability analysis.

In [ ]:
N_REPEATS = 10
all_projections_leaveout = []

for repeat in range(N_REPEATS):
    neuron_ids_leaveout = dpca_utils.get_neuron_ids(
        neuron_metadata, toRF_sessions,
        leave_out_fraction=0.1,
        rng=np.random.default_rng(repeat),
    )

    avg, tw = dpca_utils.create_dpca_matrix(
        toRF_sessions, condition_dict_states, neuron_ids_leaveout,
        state_trial_info, neuron_metadata, ephys, ephys_config,
        condition_type="states",
    )
    fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

    results   = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
    projections = dpca_utils.cross_period_projection(results, fit_avg, alignments)
    all_projections_leaveout.append(projections)

---
## Example 4 — Fit on half the trials, project onto the held-out half

Useful for the saccade-onset (response) period or any alignment.
`split_trial_info_half` splits each session independently within each state/block.

In [ ]:
half1_trial_info, half2_trial_info = dpca_utils.split_trial_info_half(
    state_trial_info, toRF_sessions, seed=0
)

neuron_ids = dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions)

# build matrices for each half
avg_fit,  tw_fit  = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_states, neuron_ids,
    half1_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states",
)
avg_test, tw_test = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_states, neuron_ids,
    half2_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states",
)

# clean fit data (strict: drop any-NaN timepoints)
fit_avg, fit_tw, _, _ = dpca_utils.clean_dpca_data(avg_fit, tw_fit, alignments)
# clean test data (lenient: drop only all-NaN timepoints)
_, _, full_avg_test, _ = dpca_utils.clean_dpca_data(avg_test, tw_test, alignments)

# fit dPCA on saccade-onset period using half-1 trials
dpca_response = dpca_utils.fit_dpca_on_alignment(fit_avg["response"], fit_tw["response"])
response_model = dpca_response[0]  # fitted dPCA model

# project held-out half-2 trials onto the fitted axes, across all periods
_, Z_baseline_heldout  = dpca_utils.dpca_transform(response_model, full_avg_test["baseline"])
_, Z_visual_heldout    = dpca_utils.dpca_transform(response_model, full_avg_test["visual"])
_, Z_cue_heldout       = dpca_utils.dpca_transform(response_model, full_avg_test["cue"])
_, Z_response_heldout  = dpca_utils.dpca_transform(response_model, full_avg_test["response"])

---
## Example 5 — Blocks instead of HMM states

Block membership comes from `glm_hmm["data"][session_id]["prob_toRF"]`:
- `prob_toRF == 50` → equal block
- `prob_toRF != 50` → unequal block

In [ ]:
condition_dict_blocks = {
    "state_values": ["equal", "unequal"],
    "coherences":   [0, 0.06, 0.2, 0.5],
    "choices":      ["awayRF", "toRF"],
}

block_trial_info = {
    "equal":   equal_block_trial_info,
    "unequal": unequal_block_trial_info,
}

neuron_ids = dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_blocks, neuron_ids,
    block_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results_blocks = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
projections_blocks  = dpca_utils.cross_period_projection(dpca_results_blocks, full_avg, alignments)

---
## Example 6 — Combining filters: exclude trash, 10% leave-out, blocks

In [ ]:
neuron_ids_combined = dpca_utils.get_neuron_ids(
    neuron_metadata, toRF_sessions,
    exclude_cell_types=["trash"],
    leave_out_fraction=0.1,
    rng=np.random.default_rng(0),
)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_blocks, neuron_ids_combined,
    block_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results_combined = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
projections_combined  = dpca_utils.cross_period_projection(dpca_results_combined, full_avg, alignments)